In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------
PROJECT_DIR = Path("/Users/subhronil/pdi/pdi_pensions-1")
DATA_DIR    = PROJECT_DIR / "regression_analysis" / "analytic samples" / "recruitment"

OUTPUT_DIR  = Path.home() / "Downloads"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_PAIRS = [
    (2008, 2009),
    (2009, 2010),
    (2010, 2011),
    (2011, 2012),
    (2012, 2013),
    (2013, 2014),
    (2014, 2015),
    (2015, 2016),
    (2016, 2017),
]

FILE_PATTERN = "analytic_sample_{yy0}{yy1}_recruitment.csv"

DV            = "joined_state_local"
TREATMENT_VAR = "illinois"
PAIR_VAR      = "pair"

WEIGHT_CANDIDATES = [
    "weight_t", "wgt_t", "wtfinl_t", "finalwgt_t",
    "weight", "wgt", "wtfinl", "finalwgt",
    "earnwt_t", "earnwt"
]

COV_TYPE = "HC1"

EXCLUDED_STATE_FIPS  = {36}
EXCLUDED_STATE_NAMES = {"ny", "new york", "new york state", "nyc", "new york city"}

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists?", DATA_DIR.exists())
if DATA_DIR.exists():
    print("CSV files found:")
    print([p.name for p in sorted(DATA_DIR.glob("*.csv"))])

# -------------------------------------------------------------------
# CONTROL VARIABLES
# -------------------------------------------------------------------
CONTINUOUS_CONTROL_CANDIDATES = [
    "age_t",
    "I(age_t**2)",
    "np.log(earnwke_t)",
]

CATEGORICAL_CONTROL_CANDIDATES = [
    "C(sex_t)",
    "C(race_t)",
    "C(ethnic_t)",
    "C(ethnicity_t)",
    "C(hispan_t)",
    "C(hispanic_t)",
    "C(hisp_t)",
    "C(grade92_t)",
    "C(educ_t)",
    "C(docc00_t)",
    "C(docc80_t)",
    "C(occ2010_t)",
    "C(occ_t)",
    "C(ind02_t)",
    "C(naics2_t)",
    "C(ind_t)",
    "C(unionmme_t)",
    "C(unioncov_t)",
]

ALL_CONTROL_CANDIDATES = CONTINUOUS_CONTROL_CANDIDATES + CATEGORICAL_CONTROL_CANDIDATES


def extract_needed_columns(terms):
    cols = set()
    for term in terms:
        for m in re.findall(r"C\(([^)]+)\)", term):
            cols.add(m.strip())
        for m in re.findall(r"np\.log\(([^)]+)\)", term):
            cols.add(m.strip())
        for expr in re.findall(r"I\(([^)]+)\)", term):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
        if not term.startswith(("C(", "I(", "np.log(")):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", term):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
    return sorted(cols)


def is_categorical_term(term):
    return term.startswith("C(")


def categorical_column_from_term(term):
    m = re.match(r"C\(([^)]+)\)", term)
    return m.group(1).strip() if m else None


def choose_one_present(terms, priority):
    present = [t for t in priority if t in terms]
    if len(present) <= 1:
        return terms
    keep = present[0]
    return [t for t in terms if (t not in priority or t == keep)]


def available_terms(df, candidate_terms):
    terms = []
    for term in candidate_terms:
        raw_cols = extract_needed_columns([term])
        if all(c in df.columns for c in raw_cols):
            terms.append(term)
    terms = choose_one_present(terms, ["C(docc00_t)", "C(docc80_t)", "C(occ2010_t)", "C(occ_t)"])
    terms = choose_one_present(terms, ["C(ind02_t)", "C(naics2_t)", "C(ind_t)"])
    terms = choose_one_present(terms, ["C(hispan_t)", "C(hispanic_t)", "C(hisp_t)", "C(ethnic_t)", "C(ethnicity_t)"])
    return terms


def build_formula(dv, controls):
    rhs_terms = [TREATMENT_VAR] + controls
    return f"{dv} ~ " + " + ".join(rhs_terms)


def detect_weight_var(df):
    for c in WEIGHT_CANDIDATES:
        if c in df.columns:
            return c
    return None


DATA_DIR: /Users/subhronil/pdi/pdi_pensions-1/regression_analysis/analytic samples/recruitment
DATA_DIR exists? True
CSV files found:
['analytic_sample_0809_recruitment.csv', 'analytic_sample_0910_recruitment.csv', 'analytic_sample_1011_recruitment.csv', 'analytic_sample_1112_recruitment.csv', 'analytic_sample_1213_recruitment.csv', 'analytic_sample_1314_recruitment.csv', 'analytic_sample_1415_recruitment.csv', 'analytic_sample_1516_recruitment.csv', 'analytic_sample_1617_recruitment.csv']


In [6]:

# -------------------------------------------------------------------
# FILE LOADING + NY REMOVAL
# -------------------------------------------------------------------

def pair_to_file(y0, y1):
    yy0 = str(y0)[-2:]
    yy1 = str(y1)[-2:]
    return DATA_DIR / FILE_PATTERN.format(yy0=yy0, yy1=yy1)


def remove_ny(df):
    before = len(df)
    mask   = pd.Series(False, index=df.index)
    used_cols = []

    # Numeric FIPS columns — use stfips_t pre-2013, stfips post-2013
    for col in ["stfips_t", "stfips"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.isin(EXCLUDED_STATE_FIPS)
            used_cols.append(col)

    # String state name columns
    for col in ["state_name_t", "state_abbrev_t", "state_t"]:
        if col in df.columns:
            vals = df[col].astype(str).str.strip().str.lower()
            mask = mask | vals.isin(EXCLUDED_STATE_NAMES) | vals.str.contains("new york", na=False)
            used_cols.append(col)

    # NYC CBSA code
    for col in ["cbsafips_t", "cbsa_t"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.eq(35620)
            used_cols.append(col)

    df2 = df.loc[~mask].copy()
    print(f"Removed NY rows: {before - len(df2):,} using columns {sorted(set(used_cols))}")
    return df2


def load_pair(y0, y1):
    path = pair_to_file(y0, y1)
    print("\n" + "=" * 90)
    print(f"PAIR {y0}-{y1}  |  Looking for: {path.name}")

    if not path.exists():
        print(f"WARNING: file not found, skipping.")
        return None

    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
    df[PAIR_VAR] = f"{str(y0)[-2:]}{str(y1)[-2:]}"

    print(f"Loaded shape: {df.shape}")
    df = remove_ny(df)
    print(f"Shape after NY removal: {df.shape}")
    print(f"Detected weight variable: {detect_weight_var(df)}")
    return df

# -------------------------------------------------------------------
# DATA CLEANING
# -------------------------------------------------------------------

def clean_categorical_series(s):
    out = s.astype("object")
    out = out.where(~pd.isna(out), "MISSING")
    out = out.astype(str).str.strip()
    out = out.replace({
        "": "MISSING", "<NA>": "MISSING", "nan": "MISSING",
        "NaN": "MISSING", "None": "MISSING", "none": "MISSING",
    })
    return out.astype("object")


def prepare_model_data(df, controls):
    weight_var = detect_weight_var(df)
    if weight_var is None:
        return None, None, "No weight variable found"

    missing = [c for c in [DV, TREATMENT_VAR] if c not in df.columns]
    if missing:
        return None, None, f"Missing required columns: {missing}"

    data = df.copy()
    data[weight_var]    = pd.to_numeric(data[weight_var],    errors="coerce")
    data[DV]            = pd.to_numeric(data[DV],            errors="coerce")
    data[TREATMENT_VAR] = pd.to_numeric(data[TREATMENT_VAR], errors="coerce")

    numeric_cols = [DV, TREATMENT_VAR, weight_var]
    for term in controls:
        if not is_categorical_term(term):
            numeric_cols.extend(extract_needed_columns([term]))
    numeric_cols = sorted(set(c for c in numeric_cols if c in data.columns))

    for c in numeric_cols:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    for term in controls:
        if is_categorical_term(term):
            c = categorical_column_from_term(term)
            if c and c in data.columns:
                data[c] = clean_categorical_series(data[c])

    if "np.log(earnwke_t)" in controls and "earnwke_t" in data.columns:
        data = data[data["earnwke_t"] > 0].copy()

    before = len(data)
    data = data.dropna(subset=numeric_cols).copy()
    data = data[data[weight_var] > 0].copy()
    after = len(data)

    if data.empty:
        return None, weight_var, "No rows left after cleaning"

    return data, weight_var, None

# -------------------------------------------------------------------
# REGRESSION — ALL COEFFICIENTS
# -------------------------------------------------------------------

def classify_term(term):
    term_l = term.lower()
    if term == "Intercept":
        return "intercept"
    if term == TREATMENT_VAR:
        return "treatment"
    if "race" in term_l:
        return "race"
    if "sex" in term_l:
        return "sex"
    if any(x in term_l for x in ["hisp", "ethnic", "ethnicity"]):
        return "ethnicity_hispanic"
    if any(x in term_l for x in ["docc", "occ"]):
        return "occupation"
    if "grade" in term_l or "educ" in term_l:
        return "education"
    if "ind" in term_l or "naics" in term_l:
        return "industry"
    if "union" in term_l:
        return "union"
    if "age" in term_l:
        return "age"
    if "earn" in term_l or "log" in term_l:
        return "earnings"
    return "other"


def run_weighted_lpm_all_terms(df, y0, y1):
    controls = available_terms(df, ALL_CONTROL_CANDIDATES)
    formula  = build_formula(DV, controls)

    data, weight_var, error = prepare_model_data(df, controls)
    if error:
        print(f"Skipping {y0}-{y1}: {error}")
        return []

    print(f"\nRunning weighted full-covariate LPM for {y0}-{y1}")
    print("Formula:", formula)
    print(f"Rows used: {len(data):,} / {len(df):,}")
    print("Weight variable:", weight_var)

    try:
        result = smf.wls(
            formula=formula,
            data=data,
            weights=data[weight_var]
        ).fit(cov_type=COV_TYPE)
    except Exception as e:
        print(f"Regression failed for {y0}-{y1}: {e}")
        return []

    conf_int = result.conf_int()
    rows = []

    for term in result.params.index:
        coef     = result.params.get(term, np.nan)
        se       = result.bse.get(term, np.nan)
        pval     = result.pvalues.get(term, np.nan)
        ci_low   = conf_int.loc[term, 0] if term in conf_int.index else np.nan
        ci_high  = conf_int.loc[term, 1] if term in conf_int.index else np.nan

        rows.append({
            "pair":                   f"{y0}-{y1}",
            "pair_code":              f"{str(y0)[-2:]}{str(y1)[-2:]}",
            "dv":                     DV,
            "term":                   term,
            "term_group":             classify_term(term),
            "coef":                   coef,
            "coef_pct_points":        coef    * 100 if pd.notna(coef)    else np.nan,
            "std_err":                se,
            "std_err_pct_points":     se      * 100 if pd.notna(se)      else np.nan,
            "p_value":                pval,
            "ci_low":                 ci_low,
            "ci_high":                ci_high,
            "ci_low_pct_points":      ci_low  * 100 if pd.notna(ci_low)  else np.nan,
            "ci_high_pct_points":     ci_high * 100 if pd.notna(ci_high) else np.nan,
            "nobs":                   int(result.nobs),
            "r_squared":              result.rsquared,
            "weight_var":             weight_var,
            "controls":               ", ".join(controls),
            "file":                   pair_to_file(y0, y1).name,
        })

    return rows

# -------------------------------------------------------------------
# RUN ALL YEAR PAIRS
# -------------------------------------------------------------------

all_coef_rows = []
loaded_pairs  = {}

for y0, y1 in YEAR_PAIRS:
    df_pair = load_pair(y0, y1)
    if df_pair is None:
        continue

    loaded_pairs[f"{y0}-{y1}"] = df_pair
    rows = run_weighted_lpm_all_terms(df_pair, y0, y1)
    all_coef_rows.extend(rows)

all_results_table = pd.DataFrame(all_coef_rows)

print("\n" + "=" * 90)
print("DONE")
print(f"Coefficient rows produced: {len(all_results_table):,}")
print(f"Year pairs successfully loaded: {list(loaded_pairs.keys())}")
display(all_results_table.head(20))

# -------------------------------------------------------------------
# TABLE 1: ILLINOIS EFFECT BY YEAR PAIR
# -------------------------------------------------------------------

illinois_effect_table = (
    all_results_table
    .query("term_group == 'treatment'")
    [["pair", "pair_code", "term", "coef", "coef_pct_points", "std_err", "std_err_pct_points",
      "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var", "file"]]
    .sort_values("pair_code")
    .reset_index(drop=True)
)
display(illinois_effect_table)

# -------------------------------------------------------------------
# TABLE 2: ALL COVARIATE EFFECTS
# -------------------------------------------------------------------

all_covariate_effects_table = (
    all_results_table
    .query("term != 'Intercept'")
    [["pair", "pair_code", "term_group", "term", "coef", "coef_pct_points", "std_err",
      "std_err_pct_points", "p_value", "ci_low_pct_points", "ci_high_pct_points",
      "nobs", "r_squared", "weight_var"]]
    .sort_values(["pair_code", "term_group", "term"])
    .reset_index(drop=True)
)
display(all_covariate_effects_table)

# -------------------------------------------------------------------
# TABLE 3: DEMOGRAPHIC, OCCUPATION, INDUSTRY, UNION, AGE, EARNINGS
# -------------------------------------------------------------------

focus_groups = [
    "treatment", "race", "sex", "ethnicity_hispanic",
    "occupation", "education", "industry", "union", "age", "earnings"
]

focus_effects_table = (
    all_results_table
    .query("term_group in @focus_groups and term != 'Intercept'")
    [["pair", "pair_code", "term_group", "term", "coef_pct_points", "std_err_pct_points",
      "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var"]]
    .sort_values(["pair_code", "term_group", "term"])
    .reset_index(drop=True)
)
display(focus_effects_table)

# -------------------------------------------------------------------
# TABLE 4: AVERAGE COEFFICIENT ACROSS YEAR PAIRS BY TERM
# -------------------------------------------------------------------

average_effects_by_term = (
    all_results_table
    .query("term != 'Intercept'")
    .groupby(["term_group", "term"], as_index=False)
    .agg(
        mean_coef=("coef", "mean"),
        mean_coef_pct_points=("coef_pct_points", "mean"),
        median_coef_pct_points=("coef_pct_points", "median"),
        num_year_pairs=("pair_code", "nunique"),
        mean_nobs=("nobs", "mean"),
    )
    .sort_values(["term_group", "term"])
    .reset_index(drop=True)
)
display(average_effects_by_term)

# -------------------------------------------------------------------
# SAVE
# -------------------------------------------------------------------

out_all_csv      = OUTPUT_DIR / "recruitment_weighted_all_covariate_coefficients.csv"
out_illinois_csv = OUTPUT_DIR / "recruitment_weighted_illinois_effects.csv"
out_focus_csv    = OUTPUT_DIR / "recruitment_weighted_demographic_occupation_effects.csv"
out_avg_csv      = OUTPUT_DIR / "recruitment_weighted_average_effects_by_term.csv"
out_xlsx         = OUTPUT_DIR / "recruitment_weighted_regression_tables.xlsx"

all_results_table.to_csv(out_all_csv, index=False)
illinois_effect_table.to_csv(out_illinois_csv, index=False)
focus_effects_table.to_csv(out_focus_csv, index=False)
average_effects_by_term.to_csv(out_avg_csv, index=False)

with pd.ExcelWriter(out_xlsx) as writer:
    illinois_effect_table.to_excel(writer,        sheet_name="illinois_effects",    index=False)
    all_covariate_effects_table.to_excel(writer,  sheet_name="all_covariates",      index=False)
    focus_effects_table.to_excel(writer,          sheet_name="demo_occ_effects",    index=False)
    average_effects_by_term.to_excel(writer,      sheet_name="avg_by_term",         index=False)
    all_results_table.to_excel(writer,            sheet_name="raw_all_terms",       index=False)

print("Saved:")
print("-", out_all_csv)
print("-", out_illinois_csv)
print("-", out_focus_csv)
print("-", out_avg_csv)
print("-", out_xlsx)


PAIR 2008-2009  |  Looking for: analytic_sample_0809_recruitment.csv
Loaded shape: (18302, 205)
Removed NY rows: 6,172 using columns ['cbsafips_t', 'state_t', 'stfips_t']
Shape after NY removal: (12130, 205)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2008-2009
Formula: joined_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(unionmme_t) + C(unioncov_t)
Rows used: 6,424 / 12,130
Weight variable: weight_t

PAIR 2009-2010  |  Looking for: analytic_sample_0910_recruitment.csv
Loaded shape: (18724, 203)
Removed NY rows: 6,257 using columns ['cbsafips_t', 'state_t', 'stfips_t']
Shape after NY removal: (12467, 203)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2009-2010
Formula: joined_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(un

,pair,pair_code,dv,term,term_group,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low,ci_high,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,controls,file
0,2008-2009,0809,joined_state_local,Intercept,intercept,-0.020336,-2.033634,0.028542,2.854175,0.476148,-0.076277,0.035604,-7.627715,3.560447,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
1,2008-2009,0809,joined_state_local,C(sex_t)[T.2],sex,0.015729,1.572888,0.005731,0.573103,0.006060,0.004496,0.026961,0.449626,2.696150,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
2,2008-2009,0809,joined_state_local,C(race_t)[T.10],race,-0.079444,-7.944377,0.040420,4.041956,0.049359,-0.158665,-0.000223,-15.866466,-0.022288,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
3,2008-2009,0809,joined_state_local,C(race_t)[T.15],race,-0.095663,-9.566299,0.067454,6.745417,0.156135,-0.227871,0.036545,-22.787074,3.654476,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
4,2008-2009,0809,joined_state_local,C(race_t)[T.2],race,0.015607,1.560698,0.011175,1.117511,0.162539,-0.006296,0.037510,-0.629583,3.750979,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
5,2008-2009,0809,joined_state_local,C(race_t)[T.3],race,0.065499,6.549852,0.067609,6.760898,0.332653,-0.067013,0.198010,-6.701266,19.800969,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
6,2008-2009,0809,joined_state_local,C(race_t)[T.4],race,-0.026686,-2.668632,0.010979,1.097892,0.015070,-0.048205,-0.005168,-4.820462,-0.516803,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
7,2008-2009,0809,joined_state_local,C(race_t)[T.5],race,-0.022526,-2.252636,0.018611,1.861096,0.226133,-0.059003,0.013950,-5.900316,1.395045,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
8,2008-2009,0809,joined_state_local,C(race_t)[T.6],race,-0.004095,-0.409491,0.016172,1.617192,0.800105,-0.035791,0.027601,-3.579129,2.760148,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
9,2008-2009,0809,joined_state_local,C(race_t)[T.7],race,-0.039675,-3.967510,0.015949,1.594929,0.012862,-0.070935,-0.008415,-7.093513,-0.841506,6424,0.068868,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv


,pair,pair_code,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,file
0,2008-2009,0809,illinois,0.001154,0.115402,0.005251,0.525094,0.826047,-0.913763,1.144566,6424,0.068868,weight_t,analytic_sample_0809_recruitment.csv
1,2009-2010,0910,illinois,0.008434,0.843377,0.005015,0.501514,0.092635,-0.139573,1.826327,6157,0.078604,weight_t,analytic_sample_0910_recruitment.csv
2,2010-2011,1011,illinois,0.009748,0.974783,0.005061,0.506125,0.054108,-0.017204,1.966770,6015,0.074036,weight_t,analytic_sample_1011_recruitment.csv
3,2011-2012,1112,illinois,-0.002283,-0.228300,0.005086,0.508644,0.653547,-1.225224,0.768625,5937,0.078359,weight_t,analytic_sample_1112_recruitment.csv
4,2012-2013,1213,illinois,0.019922,1.992160,0.005332,0.533248,0.000187,0.947013,3.037307,5934,0.079632,weight_t,analytic_sample_1213_recruitment.csv
5,2013-2014,1314,illinois,0.005565,0.556543,0.005012,0.501244,0.266859,-0.425877,1.538964,5161,0.087153,weight_t,analytic_sample_1314_recruitment.csv
6,2014-2015,1415,illinois,0.010270,1.026989,0.005595,0.559504,0.066427,-0.069619,2.123597,3836,0.160590,weight_t,analytic_sample_1415_recruitment.csv
7,2015-2016,1516,illinois,-0.004604,-0.460424,0.004645,0.464531,0.321608,-1.370888,0.450040,4435,0.145730,weight_t,analytic_sample_1516_recruitment.csv
8,2016-2017,1617,illinois,0.000723,0.072259,0.005405,0.540497,0.893648,-0.987095,1.131613,4394,0.130700,weight_t,analytic_sample_1617_recruitment.csv


,pair,pair_code,term_group,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.000004,-0.000418,0.000012,0.001244,0.736559,-0.002856,0.002019,6424,0.068868,weight_t
1,2008-2009,0809,age,age_t,0.000572,0.057182,0.001119,0.111868,0.609242,-0.162075,0.276439,6424,0.068868,weight_t
2,2008-2009,0809,earnings,np.log(earnwke_t),-0.005863,-0.586340,0.004262,0.426175,0.168878,-1.421629,0.248948,6424,0.068868,weight_t
3,2008-2009,0809,education,C(grade92_t)[T.32],-0.005525,-0.552520,0.030151,3.015130,0.854602,-6.462066,5.357025,6424,0.068868,weight_t
4,2008-2009,0809,education,C(grade92_t)[T.33],-0.008901,-0.890120,0.025715,2.571472,0.729228,-5.930112,4.149872,6424,0.068868,weight_t
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2706,2016-2017,1617,sex,C(sex_t)[T.2],0.000162,0.016245,0.005396,0.539610,0.975984,-1.041372,1.073861,4394,0.130700,weight_t
2707,2016-2017,1617,treatment,illinois,0.000723,0.072259,0.005405,0.540497,0.893648,-0.987095,1.131613,4394,0.130700,weight_t
2708,2016-2017,1617,union,C(unioncov_t)[T.2.0],-0.026738,-2.673799,0.038110,3.811036,0.482932,-10.143292,4.795693,4394,0.130700,weight_t
2709,2016-2017,1617,union,C(unioncov_t)[T.MISSING],0.006233,0.623335,0.019588,1.958767,0.750312,-3.215778,4.462447,4394,0.130700,weight_t


,pair,pair_code,term_group,term,coef_pct_points,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.000418,0.001244,0.736559,-0.002856,0.002019,6424,0.068868,weight_t
1,2008-2009,0809,age,age_t,0.057182,0.111868,0.609242,-0.162075,0.276439,6424,0.068868,weight_t
2,2008-2009,0809,earnings,np.log(earnwke_t),-0.586340,0.426175,0.168878,-1.421629,0.248948,6424,0.068868,weight_t
3,2008-2009,0809,education,C(grade92_t)[T.32],-0.552520,3.015130,0.854602,-6.462066,5.357025,6424,0.068868,weight_t
4,2008-2009,0809,education,C(grade92_t)[T.33],-0.890120,2.571472,0.729228,-5.930112,4.149872,6424,0.068868,weight_t
...,...,...,...,...,...,...,...,...,...,...,...,...
2706,2016-2017,1617,sex,C(sex_t)[T.2],0.016245,0.539610,0.975984,-1.041372,1.073861,4394,0.130700,weight_t
2707,2016-2017,1617,treatment,illinois,0.072259,0.540497,0.893648,-0.987095,1.131613,4394,0.130700,weight_t
2708,2016-2017,1617,union,C(unioncov_t)[T.2.0],-2.673799,3.811036,0.482932,-10.143292,4.795693,4394,0.130700,weight_t
2709,2016-2017,1617,union,C(unioncov_t)[T.MISSING],0.623335,1.958767,0.750312,-3.215778,4.462447,4394,0.130700,weight_t


,term_group,term,mean_coef,mean_coef_pct_points,median_coef_pct_points,num_year_pairs,mean_nobs
0,age,I(age_t ** 2),-0.000004,-0.000383,-0.000756,9,5365.888889
1,age,age_t,0.000301,0.030126,0.069942,9,5365.888889
2,earnings,np.log(earnwke_t),-0.003457,-0.345743,-0.363907,9,5365.888889
3,education,C(grade92_t)[T.32],-0.011296,-1.129639,-0.552520,9,5365.888889
4,education,C(grade92_t)[T.33],-0.009262,-0.926162,-0.890120,9,5365.888889
...,...,...,...,...,...,...,...
333,sex,C(sex_t)[T.2],0.006045,0.604532,0.793035,9,5365.888889
334,treatment,illinois,0.005436,0.543643,0.556543,9,5365.888889
335,union,C(unioncov_t)[T.2.0],-0.007644,-0.764360,-0.490142,9,5365.888889
336,union,C(unioncov_t)[T.MISSING],0.014300,1.430040,2.509057,9,5365.888889


Saved:
- /Users/subhronil/Downloads/recruitment_weighted_all_covariate_coefficients.csv
- /Users/subhronil/Downloads/recruitment_weighted_illinois_effects.csv
- /Users/subhronil/Downloads/recruitment_weighted_demographic_occupation_effects.csv
- /Users/subhronil/Downloads/recruitment_weighted_average_effects_by_term.csv
- /Users/subhronil/Downloads/recruitment_weighted_regression_tables.xlsx
